# 03 — Modeling results

**Goal:** pick a model family for IEEE-CIS fraud detection, settle how we handle the ~3.5% class
imbalance, and tune the winner, using **cross-validation on the training set only**. The outer time-based
test set (last 20% of time) is not touched here; that's Stage 6.

**This notebook only reports.** All training happens in `src/fraud_detection/run_experiments.py`:

```
python -m src.fraud_detection.run_experiments
```

That script cross-validates five pipelines, tunes the one with the best PR-AUC with `RandomizedSearchCV`, logs
every run to MLflow (experiment `ieee-fraud-detection`), and saves `reports/results/cv_results.json`,
`reports/results/tuning_results.json` and `models/final_pipeline.joblib`. Everything below is read from
those two JSON files.

How the numbers were produced:

* **Folds** come from `TimeAwareStratifiedSplit`. It uses an expanding window: each fold trains on the past and
  validates on the next block of time, with at least `min_pos` frauds per validation block. No random
  `StratifiedKFold` is used, because it would let the model train on the future.
* **Each fold refits a fresh copy of the whole pipeline** on that fold's training rows, so scaling and SMOTE
  never see validation data.
* **Metrics are for the fraud class**: precision, recall and F1 at the saved threshold, plus ROC-AUC and PR-AUC
  (average precision). Accuracy is never used, because predicting "legit" for everything already scores ~96.5%.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
ROOT = Path("..").resolve()
RESULTS_DIR = ROOT / "reports" / "results"
FIGURES_DIR = ROOT / "reports" / "figures"

cv = json.loads((RESULTS_DIR / "cv_results.json").read_text(encoding="utf-8"))
tuning = json.loads((RESULTS_DIR / "tuning_results.json").read_text(encoding="utf-8"))
meta = cv["meta"]

# dry-run (subsampled) results never overwrite the committed report figures
SAVE_FIGS = meta["sample_n"] is None

METRICS = ["precision", "recall", "f1", "roc_auc", "pr_auc"]
METRIC_NAMES = {"precision": "Precision", "recall": "Recall", "f1": "F1", "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"}
LABELS = {k: v["label"] for k, v in cv["models"].items()}
# one fixed color per model across every chart (validated categorical palette, slots 1-5)
COLORS = dict(zip(["logreg", "logreg_smote", "rf", "xgb", "lgbm"],
                  ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))

if meta["sample_n"] is not None:
    print(f"⚠ DRY RUN: these results come from a {meta['sample_n']:,}-row random subsample, "
          f"not the full training set. Don't draw conclusions from them.")
print(f"Results produced {meta['created_at']} | {meta['n_rows']:,} rows x {meta['n_features']} features | "
      f"fraud rate {meta['fraud_rate']:.2%} | threshold {meta['threshold']} | seed {meta['seed']}")

### The CV folds

Each fold trains on all earlier time blocks and validates on the next one, so the training set grows from fold to fold.

In [ ]:
pd.DataFrame(meta["folds"], index=pd.RangeIndex(1, len(meta["folds"]) + 1, name="fold")).assign(
    val_fraud_rate=lambda d: (d["val_frauds"] / d["val_rows"]).map("{:.2%}".format))

## Model comparison, sorted by PR-AUC

| Key | Pipeline | Imbalance handling |
|---|---|---|
| `logreg` | StandardScaler → LogisticRegression | `class_weight="balanced"` (**baseline**) |
| `logreg_smote` | StandardScaler → SMOTE → LogisticRegression (imblearn Pipeline) | SMOTE oversampling |
| `rf` | RandomForest | `class_weight="balanced"` |
| `xgb` | XGBoost | `scale_pos_weight = neg/pos` from the training labels |
| `lgbm` | LightGBM | `class_weight="balanced"` |

**Why sort by PR-AUC and not ROC-AUC (or accuracy)?** Only ~3.5% of transactions are fraud. ROC-AUC measures
true-positive rate against *false-positive rate*, and with ~27 legit rows for every fraud, even many false alarms
make only a small FPR. That makes ROC-AUC look flattering and squeezes the gap between models. PR-AUC (average
precision) is built from the **precision and recall of the fraud class**. It asks "of the transactions we'd flag,
how many are actually fraud?" across all thresholds, so false alarms hurt it directly. Its no-skill baseline is the
fraud rate (~0.035), not 0.5, so the real headroom is visible.

Precision, recall and F1 use a fixed 0.5 threshold, which means different things for differently calibrated models.
For example, a `class_weight="balanced"` logistic regression pushes probabilities up, trading precision for recall.
They're shown for completeness; the threshold itself gets tuned later.

In [ ]:
def fmt(m):
    return f"{m['mean']:.3f} ({m['std']:.3f})"

order = sorted(cv["models"], key=lambda k: cv["models"][k]["metrics"]["pr_auc"]["mean"], reverse=True)
comparison = pd.DataFrame({
    LABELS[k]: {**{METRIC_NAMES[m]: fmt(cv["models"][k]["metrics"][m]) for m in METRICS},
                "CV time (min)": f"{cv['models'][k]['cv_seconds'] / 60:.1f}"}
    for k in order
}).T
print(f"mean (std) over {len(meta['folds'])} time-ordered folds; no-skill PR-AUC = fraud rate = {meta['fraud_rate']:.3f}")
comparison

In [ ]:
def grouped_bars(ax, metrics_by_key, keys, labels, colors, metrics=("precision", "recall", "f1", "pr_auc")):
    # metrics_by_key[k][metric] = {"mean", "std", "folds"}, the saved JSON shape
    n = len(keys)
    width = 0.8 / n
    x = np.arange(len(metrics))
    for i, k in enumerate(keys):
        ax.bar(x + (i - (n - 1) / 2) * width,
               [metrics_by_key[k][m]["mean"] for m in metrics], width * 0.92,
               yerr=[metrics_by_key[k][m]["std"] for m in metrics], capsize=3,
               color=colors[k], edgecolor="white", linewidth=1, label=labels[k],
               error_kw={"elinewidth": 1, "ecolor": "#52514e"})
    ax.set_xticks(x, [METRIC_NAMES[m] for m in metrics])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score (fraud class)")
    ax.grid(axis="x", visible=False)


model_metrics = {k: v["metrics"] for k, v in cv["models"].items()}

fig, ax = plt.subplots(figsize=(11, 5))
grouped_bars(ax, model_metrics, order, LABELS, COLORS)
ax.axhline(meta["fraud_rate"], color="#52514e", linestyle=":", linewidth=1,
           label=f"no-skill PR-AUC = fraud rate ({meta['fraud_rate']:.3f})")
ax.set_title("Cross-validated performance by model (sorted by PR-AUC, error bars ±1 std over folds)", fontsize=12)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.08), frameon=False)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(FIGURES_DIR / "07_model_comparison.png", bbox_inches="tight")
plt.show()

## Imbalance handling: `class_weight` vs SMOTE

This compares the same model (scaled logistic regression) on the same folds. The only difference is how the imbalance
is handled. `class_weight="balanced"` reweights the loss so each fraud counts ~27×. SMOTE instead generates synthetic
fraud rows by interpolating between real ones until the classes are 50/50. The left panel shows all metrics. The right
panel shows PR-AUC fold by fold, to see whether any gap is consistent or just noise.

In [ ]:
lr_keys = ["logreg", "logreg_smote"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})

grouped_bars(axes[0], model_metrics, lr_keys, LABELS, COLORS, metrics=METRICS)
axes[0].set_title("All metrics (mean ± std over folds)")
axes[0].legend(frameon=False)

folds = np.arange(1, len(meta["folds"]) + 1)
for k in lr_keys:
    axes[1].plot(folds, model_metrics[k]["pr_auc"]["folds"], marker="o", markersize=8, linewidth=2,
                 color=COLORS[k], label=LABELS[k])
axes[1].set_xticks(folds)
axes[1].set_xlabel("Fold (later folds = later in time, more training data)")
axes[1].set_ylabel("PR-AUC")
axes[1].set_title("PR-AUC per fold")
axes[1].legend(frameon=False)

plt.suptitle("Logistic regression: class_weight vs SMOTE", fontsize=13)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(FIGURES_DIR / "08_class_weight_vs_smote.png", bbox_inches="tight")
plt.show()

delta = np.array(model_metrics["logreg_smote"]["pr_auc"]["folds"]) - np.array(model_metrics["logreg"]["pr_auc"]["folds"])
print(f"PR-AUC, SMOTE minus class_weight: mean {delta.mean():+.4f}, SMOTE better on {(delta > 0).sum()}/{len(delta)} folds")
print(f"CV time: class_weight {cv['models']['logreg']['cv_seconds'] / 60:.1f} min "
      f"vs SMOTE {cv['models']['logreg_smote']['cv_seconds'] / 60:.1f} min")

### Conclusion: class_weight vs SMOTE

_To be written from the full-run numbers._

## Tuning the winning family

The family with the highest mean CV PR-AUC was tuned with `RandomizedSearchCV`, under two constraints:

* **Same folds.** It used the same `TimeAwareStratifiedSplit` folds passed as `cv=`, so tuning respects time order too.
* **Same metric.** It used `scoring="average_precision"`, the metric that picked the family.

The search only scores PR-AUC, so the script reran the tuned configuration through the same CV to get every metric
like-for-like.

**Caveat:** these folds were also used to *choose* the tuned configuration, so the tuned numbers are slightly
optimistic (selection bias across the candidates). The unbiased check is the untouched time-based test set in Stage 6.

In [ ]:
fam = tuning["family"]
print(f"Winning family: {tuning['label']} | {tuning['meta']['n_iter']} candidates | "
      f"search took {tuning['search_seconds'] / 60:.1f} min | best search PR-AUC {tuning['best_search_pr_auc']:.4f}")
pd.Series(tuning["best_params"], name="best value", dtype=object).to_frame()  # object keeps ints as ints

In [ ]:
before_after = {"untuned": tuning["untuned"], "tuned": tuning["tuned"]}
ba_labels = {"untuned": f"{tuning['label']}: defaults", "tuned": f"{tuning['label']}: tuned"}
ba_colors = {"untuned": "#a9a8a2", "tuned": COLORS.get(fam, "#2a78d6")}
shown = ["precision", "recall", "f1", "pr_auc"]

fig, ax = plt.subplots(figsize=(9, 4.5))
grouped_bars(ax, before_after, ["untuned", "tuned"], ba_labels, ba_colors, metrics=shown)
for i, m in enumerate(shown):
    u, t = before_after["untuned"][m], before_after["tuned"][m]
    ax.text(i, max(u["mean"] + u["std"], t["mean"] + t["std"]) + 0.03,
            f"Δ {t['mean'] - u['mean']:+.3f}", ha="center", fontsize=10, color="#0b0b0b")
ax.set_title(f"{tuning['label']}: before vs after RandomizedSearchCV", fontsize=13)
ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.08), frameon=False)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(FIGURES_DIR / "09_tuning_before_after.png", bbox_inches="tight")
plt.show()

pd.DataFrame({ba_labels[k]: {METRIC_NAMES[m]: fmt(r[m]) for m in METRICS} for k, r in before_after.items()})

## Summary

_To be written from the full-run numbers._